# 🧹 NOTEBOOK 4b — Test Set Correction (removing augmented duplicates)

**Why this notebook?**
The intra-source pre-augmentation (`00_extract_lidc_idri.ipynb`, `00_extract_lndb.ipynb`, suffixes `_rot90/_rot180/_rot270/_flip/_bright`)
happens **before** consolidation (`01_dataset_building.ipynb`) and the patient-level split. The split therefore only
guarantees the absence of *inter*-split leakage (a patient appears in only one of train/val/test) — not *intra*-split
independence: `test.csv` contains both original patches and their augmented copies from the same patient.

This notebook:
1. Reloads `test.csv` and reproduces **exactly** the same deterministic shuffle used in `03_train_cbam_cnn.ipynb`
   (`df.sample(frac=1, random_state=42)`), to recover the order in which `X_test.npy` / `y_test.npy` were saved
   (no path array was saved at the time — we do it here, for next time).
2. Computes the `is_augmented` mask from the filename.
3. Filters `X_test.npy` / `y_test.npy` for each of the 4 architectures and saves
   `X_test_corrected.npy` / `y_test_corrected.npy` + a manifest CSV.
4. **Validates** the result against the expected counts (manuscript Table 3: 2,664 = 2,328 / 181 / 155).
   If validation fails → switch to Option B (clean re-extraction, bottom cell).

In [1]:
# =============================================
# CELL 1: CONFIG
# =============================================
import os, json
import numpy as np
import pandas as pd
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# ============================================================
# 🔧 USER CONFIGURATION — edit PROJECT_ROOT for your own setup
# ============================================================
PROJECT_ROOT = "/content/drive/MyDrive/lung_nodule_classification"

MASTER_CSV_DIR = f"{PROJECT_ROOT}/master_dataset"
FEATURES_DIR   = f"{PROJECT_ROOT}/features"

TEST_CSV      = os.path.join(MASTER_CSV_DIR, "test.csv")
ARCHITECTURES = ['ResNet50V2', 'DenseNet201', 'EfficientNetB3', 'EfficientNetV2L']
RANDOM_SEED   = 42

# Must be EXACTLY the same list as in 00_extract_lidc_idri.ipynb / 00_extract_lndb.ipynb
AUG_SUFFIXES = ['_rot90', '_rot180', '_rot270', '_flip', '_bright']

print("✅ Config loaded.")


Mounted at /content/drive
✅ Config loaded.


In [2]:
# =============================================
# CELL 2: REPRODUCE THE EXACT ORDER OF build_dataset() (Notebook 3)
# =============================================
# build_dataset(..., shuffle=True) does:
#     df = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
# then iterates WITHOUT further reshuffling. X_test.npy / y_test.npy are therefore
# in THIS order.

df_test = pd.read_csv(TEST_CSV)
n_raw = len(df_test)
print(f"Raw test.csv: {n_raw} rows")

df_test_shuffled = df_test.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# Mask of augmented duplicates (detects the suffix in the filename)
def is_augmented(path):
    fname = os.path.basename(path)
    return any(suf in fname for suf in AUG_SUFFIXES)

df_test_shuffled['is_augmented'] = df_test_shuffled['path'].apply(is_augmented)
keep_mask = ~df_test_shuffled['is_augmented'].values

n_kept = keep_mask.sum()
print(f"After removing augmented duplicates: {n_kept} / {n_raw} rows kept")
print()
print("Class distribution (kept):")
print(df_test_shuffled.loc[keep_mask, 'label'].value_counts().sort_index())

# --- Validation against the manuscript (Table 3: 2664 = 2328/181/155) -------
expected = {0: 2328, 1: 181, 2: 155}
counts = df_test_shuffled.loc[keep_mask, 'label'].value_counts().sort_index().to_dict()
ok = (n_kept == 2664) and all(counts.get(k) == v for k, v in expected.items())
print()
if ok:
    print("✅ VALIDATION OK — counts match the manuscript's Table 3 exactly.")
    print("   You can proceed with the next cells (Option A).")
else:
    print("⚠️  VALIDATION FAILED — the reconstructed order does not match the expected counts.")
    print("   Do NOT use these indices to filter X_test.npy. Use Option B at the bottom of this")
    print("   notebook instead (clean re-extraction from the .keras checkpoints, independent of")
    print("   the shuffle order).")
    print(f"   Expected: {{0: 2328, 1: 181, 2: 155}} (total 2664) | Got: {counts} (total {n_kept})")


Raw test.csv: 5528 rows
After removing augmented duplicates: 2664 / 5528 rows kept

Class distribution (kept):
label
0    2328
1     181
2     155
Name: count, dtype: int64

✅ VALIDATION OK — counts match the manuscript's Table 3 exactly.
   You can proceed with the next cells (Option A).


In [3]:
# =============================================
# CELL 3: FILTER X_test.npy / y_test.npy FOR EACH ARCHITECTURE
# =============================================
# ⚠️ Only run this cell if CELL 2 printed "VALIDATION OK".

manifest_rows = []

for arch in ARCHITECTURES:
    feat_dir = os.path.join(FEATURES_DIR, arch)
    x_path = os.path.join(feat_dir, 'X_test.npy')
    y_path = os.path.join(feat_dir, 'y_test.npy')

    if not (os.path.exists(x_path) and os.path.exists(y_path)):
        print(f"❌ {arch}: files not found, skipping.")
        continue

    X_test = np.load(x_path)
    y_test = np.load(y_path)

    if len(X_test) != len(df_test_shuffled):
        print(f"❌ {arch}: X_test length ({len(X_test)}) != shuffled test.csv length "
              f"({len(df_test_shuffled)}) — reconstructed order does not match. "
              f"Use Option B for this architecture.")
        continue

    # Extra sanity check: labels should match
    label_match = np.mean(y_test == df_test_shuffled['label'].values)
    if label_match < 0.999:
        print(f"⚠️  {arch}: only {label_match*100:.1f}% of labels match — "
              f"alignment looks suspect, verify before using this result.")

    X_test_corrected = X_test[keep_mask]
    y_test_corrected = y_test[keep_mask]

    np.save(os.path.join(feat_dir, 'X_test_corrected.npy'), X_test_corrected)
    np.save(os.path.join(feat_dir, 'y_test_corrected.npy'), y_test_corrected)

    print(f"✅ {arch:16s}: {len(X_test)} → {len(X_test_corrected)} (label match check: "
          f"{label_match*100:.1f}%)")

    manifest_rows.append({
        'architecture': arch,
        'n_raw': len(X_test),
        'n_corrected': len(X_test_corrected),
        'label_alignment_check': round(float(label_match), 4),
    })

manifest_df = pd.DataFrame(manifest_rows)
manifest_path = os.path.join(FEATURES_DIR, 'test_set_correction_manifest.csv')
manifest_df.to_csv(manifest_path, index=False)

# Also save the corrected test.csv (useful for any future error analysis)
df_test_shuffled.loc[keep_mask].drop(columns=['is_augmented']).to_csv(
    os.path.join(MASTER_CSV_DIR, 'test_corrected.csv'), index=False
)

print(f"\n💾 Manifest saved: {manifest_path}")
print(f"💾 test_corrected.csv saved: {os.path.join(MASTER_CSV_DIR, 'test_corrected.csv')}")
print(manifest_df.to_string(index=False))


✅ ResNet50V2      : 5528 → 2664 (label match check: 100.0%)
✅ DenseNet201     : 5528 → 2664 (label match check: 100.0%)
✅ EfficientNetB3  : 5528 → 2664 (label match check: 100.0%)
✅ EfficientNetV2L : 5528 → 2664 (label match check: 100.0%)

💾 Manifest saved: /content/drive/MyDrive/Journal_Work_2026/Features_v2/test_set_correction_manifest.csv
💾 test_corrected.csv saved: /content/drive/MyDrive/Journal_Work_2026/Master_Dataset_Final/test_corrected.csv
   architecture  n_raw  n_corrected  label_alignment_check
     ResNet50V2   5528         2664                    1.0
    DenseNet201   5528         2664                    1.0
 EfficientNetB3   5528         2664                    1.0
EfficientNetV2L   5528         2664                    1.0


---
## Next step

Once this notebook has run successfully, go to **`05_ml_classifiers.ipynb`**, whose `load_features()`
(Cell 3) already loads `X_test_corrected.npy` / `y_test_corrected.npy` produced here. Just run it from
the top: confusion matrices, heatmaps, LaTeX table, feature importance — everything regenerates
automatically with the correct numbers, no other change needed.

---
## Option B — Clean re-extraction (if the Cell 2 validation fails)

More robust (does not depend on `df.sample()` being exactly reproducible across environments), but requires
reloading each `.keras` checkpoint and re-running a forward pass over the test set (fast: no retraining
involved). Adapt this skeleton from the "RECOVERY" cell already present in `03_train_cbam_cnn.ipynb`
(Cell 16), simply adding `shuffle=False` and returning the paths alongside the features:

```python
# In build_dataset(), add path output:
def build_dataset_with_paths(csv_path, img_size, batch_size):
    df = pd.read_csv(csv_path)  # NO .sample() -> order = CSV order
    paths = df['path'].values
    ds, n = build_dataset(csv_path, img_size, batch_size, shuffle=False, cache=False)
    return ds, n, paths  # paths is already in the same order as the CSV

# Then for each architecture:
#   1. model = keras.models.load_model(f'{MODELS_DIR}/CBAM_{arch}_best.keras', ...)
#   2. feat_extractor = build_feature_extractor(model)
#   3. test_ds, n_test, test_paths = build_dataset_with_paths(TEST_CSV, cfg['img_size'], cfg['batch_size'])
#   4. X_test, y_test = extract_features(feat_extractor, test_ds, n_test, cfg['batch_size'])
#   5. is_aug = np.array([any(s in os.path.basename(p) for s in AUG_SUFFIXES) for p in test_paths])
#   6. Save X_test[~is_aug], y_test[~is_aug]  ->  guaranteed aligned, no assumption about order.
```